# 02 · Data cleaning

**Goal:** turn the raw `Combined_dataset.csv` into **one** clean, split, model-ready file: `data/processed/final_dataset.csv`.

| | |
|---|---|
| **Input** | `data/raw/Combined_dataset.csv` (2.83M rows, 5 countries, not committed) |
| **Output** | `data/processed/final_dataset.csv` (the only processed file, committed) and `docs/cleaning_log.md` |
| **Code** | all logic lives in `src/data.py` (`build_final_dataset`); this notebook runs it and shows the evidence |
| **Reproduce** | put `Combined_dataset.csv` in `data/raw/`, then run all cells |

**What we found in the raw data (the reason for each step below):**
1. Many rows are **effluent / sewage / drainage** water, not surface water.
2. Most parameters contain a **repeated filled-in value** for each country and water type (the dataset authors filled gaps), so those are not real measurements.
3. Some values are **physically impossible** (temperature above 40 °C, pH outside 0–14, ...) and there are **duplicate rows**.
4. The target (CCME index) is **calculated from the same 8 parameters**, so we must split and select features carefully (handled in `03_features`).

## 1. Setup

Import the helper functions from `src/data.py` and set the file paths.

In [1]:
import os, sys
sys.path.append("..")                      # lets us import from src/

import numpy as np
import pandas as pd

from src.data import (
    load_combined_for_countries, clean_column_names, build_final_dataset,
    write_cleaning_log_md, DEFAULT_COUNTRIES, DEFAULT_COUNTRY_CAPS, PARAM_COLS,
)

RAW_PATH = "../data/raw/Combined_dataset.csv"
OUT_PATH = "../data/processed/final_dataset.csv"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 110)

## 2. Look at the raw data

Load the four countries we use (China has a single monitoring site, so a site split is impossible there) and see how many rows each country and water type has.

In [2]:
raw = clean_column_names(load_combined_for_countries(RAW_PATH, DEFAULT_COUNTRIES))
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
display(raw.head(3))
display(pd.crosstab(raw["waterbody_type"], raw["country"], margins=True))

2,781,980 rows x 14 columns


,country,site_id,waterbody_type,date,ammonia_mg_l,bod_mg_l,dissolved_oxygen_mg_l,orthophosphate_mg_l,ph,temperature_c,nitrogen_mg_l,nitrate_mg_l,target_ccme_value,target_ccme_class
0,Canada,SE649035-145565,River,12-01-1974,0.059248,1.30,8.15,0.011917,8.07500,9.885,0.343917,11.73155,100.0,Excellent
1,Canada,SE649035-145565,River,12-01-1975,0.039821,1.38,7.80,0.009417,7.73333,10.150,0.449083,11.82009,100.0,Excellent
2,Canada,SE649035-145565,River,12-01-1976,0.031341,2.23,7.80,0.011000,7.46667,10.235,0.220750,14.87472,100.0,Excellent


country,Canada,England,Ireland,USA,All
waterbody_type,,,,,
Canal,0,28574,0,0,28574
Coastal,2,0,2817,0,2819
Drainage,0,10205,0,0,10205
Effluent,0,601550,0,0,601550
Estuarine,0,49375,0,0,49375
Lake,602,33771,24465,94765,153603
Marine,0,309,0,22853,23162
River,3288,1349576,203519,296196,1852579
Sea Water,0,32061,0,0,32061


## 3. Evidence of the problems

### 3.1 Not all water is surface water
Effluent, sewage and drainage are discharges, not rivers or lakes. Share of such rows per country:

In [3]:
non_surface = raw["waterbody_type"].isin(["Effluent", "Sewage", "Drainage"])
display(non_surface.groupby(raw["country"]).mean().mul(100).round(1)
        .rename("% effluent / sewage / drainage rows").to_frame())

,% effluent / sewage / drainage rows
country,
Canada,0.0
England,29.8
Ireland,0.0
USA,0.0


### 3.2 Filled-in values
For a real measurement, no single value should dominate. Example: England, river water. The most common value of almost every parameter makes up a huge share of rows, and is far more common than the second most common value. These are placeholders, not measurements.

In [4]:
eng = raw[(raw["country"] == "England") & (raw["waterbody_type"] == "River")]
rows = []
for c in PARAM_COLS:
    vc = eng[c].round(6).value_counts(normalize=True)
    rows.append({"parameter": c, "most common value": vc.index[0],
                 "share of rows %": round(100 * vc.iloc[0], 1),
                 "2nd most common value %": round(100 * vc.iloc[1], 1)})
display(pd.DataFrame(rows))

,parameter,most common value,share of rows %,2nd most common value %
0,ammonia_mg_l,0.03,34.5,3.3
1,bod_mg_l,2.70,43.9,11.1
2,dissolved_oxygen_mg_l,10.20,23.0,7.7
3,orthophosphate_mg_l,0.02,9.9,9.7
4,ph,7.78,13.8,4.0
5,temperature_c,11.46,7.3,0.8
6,nitrogen_mg_l,5.00,7.5,4.5
7,nitrate_mg_l,4.50,18.4,2.8


Free the memory before running the pipeline (it reloads the file itself).

In [5]:
del raw, eng, non_surface

## 4. Run the cleaning pipeline
`build_final_dataset` does every step in order and records each one in a log:

1. keep chosen countries · 2. parse dates · 3. drop effluent/sewage/drainage · 4. keep 2000 onwards · 5. drop duplicates · 6. set impossible values to NaN · 7. set filled-in placeholder values to NaN · 8. drop rows with too few real values · 9. tidy names and site ids · 10. sample whole sites per country (GitHub size) · 11. create the station split and the time split · 12. save.

In [6]:
final, log, audit = build_final_dataset(RAW_PATH, OUT_PATH, DEFAULT_COUNTRIES, DEFAULT_COUNTRY_CAPS)
print("saved:", os.path.abspath(OUT_PATH))

saved: /sessions/rcw-01ewwtexhtd5fc7lnrgniry6/mnt/water-quality-prediction/data/processed/final_dataset.csv


## 5. Cleaning log
Every decision with the real number of rows or values affected. This table is also saved to `docs/cleaning_log.md`.

In [7]:
log_df = pd.DataFrame(log)
display(log_df)

header = (
    "# Data cleaning log\n\n"
    "- **Raw input:** `data/raw/Combined_dataset.csv` (England, USA, Ireland, Canada)\n"
    "- **Output:** `data/processed/final_dataset.csv`\n"
    "- **Code:** `build_final_dataset()` in `src/data.py`, run from `notebooks/02_cleaning.ipynb`\n"
    "- Generated automatically, so every number below is from the real run (seed 42)."
)
# which filled-in values were found for the three model inputs (grouped by country)
inputs = audit[audit["parameter"].isin(["dissolved_oxygen_mg_l", "ph", "temperature_c"])]
placeholder_table = (inputs.groupby(["parameter", "country", "placeholder_value"])
    .agg(rows_flagged=("rows_flagged", "sum"), water_types=("waterbody_type", lambda s: ", ".join(sorted(s))))
    .reset_index().sort_values(["parameter", "country", "rows_flagged"], ascending=[True, True, False]))
lines = ["## Filled-in values found for the model inputs", "",
         "Values repeated for a large share of a country/water-type group (and far more common than any other value) are placeholders, not measurements. They were set to NaN. Same pattern was found and removed for the five held-out parameters (counts in the table above).", "",
         "| Parameter | Country | Repeated value | Rows set to NaN | Water types |", "| :--- | :--- | ---: | ---: | :--- |"]
for r in placeholder_table.itertuples():
    lines.append(f"| {r.parameter} | {r.country} | {r.placeholder_value:g} | {r.rows_flagged:,} | {r.water_types} |")
write_cleaning_log_md(log, "../docs/cleaning_log.md", header, footer="\n".join(lines))
print("wrote docs/cleaning_log.md")

,Issue found,How many rows/values,Action,Why
0,Rows loaded for the chosen countries,"2,781,980 {'England': 2129198, 'USA': 413814, 'Ireland': 235019, 'Canada': 3949}","Kept countries ['England', 'USA', 'Ireland', 'Canada'] from Combined_dataset.csv","Other countries are not used (China has a single site, so no site split is possible)."
1,Unparseable dates (dd-mm-yyyy),0,None found,Date is needed for the time split and seasonal features.
2,"Non-surface water types (effluent, sewage, drainage)","635,532 {'Effluent': 601550, 'Sewage': 23777, 'Drainage': 10205}",Dropped rows,The task is surface-water quality; effluent/sewage has a different chemistry and was ~28% of England.
3,Records before 2000,243089,Dropped rows,England only has data from 2000; older USA/Ireland records used different methods and are sparse.
4,Exact duplicate rows across all columns,7277,Removed duplicate rows and reset the index,Exact duplicates can over-weight repeated observations and complicate validation.
5,"ph physical bounds check [0.0, 14.0]",503,Set out-of-range values to NaN; retained rows,Outside the physical pH scale; instrument error
6,"temperature_c physical bounds check [-5.0, 40.0]",222,Set out-of-range values to NaN; retained rows,Surface water outside -5..40 C is a unit error (e.g. Fahrenheit) or a fault
7,"dissolved_oxygen_mg_l physical bounds check [0.0, 25.0]",0,No out-of-range values found,Above ~25 mg/L is not physically possible in surface water
8,"bod_mg_l physical bounds check [0.0, 150.0]",3564,Set out-of-range values to NaN; retained rows,"Above 150 mg/L looks like effluent, not surface water"
9,"ammonia_mg_l physical bounds check [0.0, 100.0]",647,Set out-of-range values to NaN; retained rows,Negative or extreme concentrations are errors


wrote docs/cleaning_log.md


### 5.1 Which values were filled-in placeholders?
The repeated values found for the three model inputs (dissolved oxygen, pH, temperature) are listed at the bottom of `docs/cleaning_log.md`. Largest ones:

In [8]:
display(audit[audit["parameter"].isin(["dissolved_oxygen_mg_l", "ph", "temperature_c"])]
        .sort_values("rows_flagged", ascending=False).head(12))

,country,waterbody_type,parameter,pass_no,placeholder_value,share_of_remaining_pct,rows_flagged
65,England,River,dissolved_oxygen_mg_l,1,10.200000,22.9,308958
67,England,River,ph,1,7.780000,13.8,185984
66,England,River,dissolved_oxygen_mg_l,2,11.684103,10.0,104073
68,England,River,temperature_c,1,11.460000,7.3,98469
97,Ireland,River,dissolved_oxygen_mg_l,1,11.684103,37.4,76170
133,USA,River,ph,1,7.800000,83.5,61443
130,USA,River,dissolved_oxygen_mg_l,1,9.870000,57.8,42547
112,USA,Lake,dissolved_oxygen_mg_l,1,9.870000,55.8,39007
115,USA,Lake,ph,1,7.800000,54.7,38258
116,USA,Lake,temperature_c,1,11.100000,54.3,37960


## 6. Check the result

### 6.1 Size and composition
Rows, monitoring sites and how much of each target is stuck at the maximum score of 100, per country.

In [9]:
display(final.groupby("country").agg(
    rows=("site_id", "size"),
    sites=("site_id", "nunique"),
    target_mean=("target_ccme_value", "mean"),
    pct_target_100=("target_ccme_value", lambda s: 100 * (s >= 99.999).mean()),
).round(1))
print("total:", f"{len(final):,}", "rows,", f"{final['site_id'].nunique():,}", "sites")

,rows,sites,target_mean,pct_target_100
country,,,,
Canada,2600,1742,90.9,35.4
England,180082,4413,87.0,28.9
Ireland,70282,3691,97.8,81.6
USA,6810,94,95.5,45.9


total: 259,774 rows, 9,940 sites


### 6.2 Missing values after cleaning
Placeholders were set to NaN, so these gaps are honest. They are imputed later using **training rows only** (in `src/features.py`).

In [10]:
display(final.groupby("country")[PARAM_COLS].apply(lambda g: g.isna().mean() * 100).round(1))

,ammonia_mg_l,bod_mg_l,dissolved_oxygen_mg_l,orthophosphate_mg_l,ph,temperature_c,nitrogen_mg_l,nitrate_mg_l
country,,,,,,,,
Canada,1.0,2.7,13.3,19.3,26.7,2.4,76.5,1.5
England,31.7,53.0,19.9,1.0,3.5,0.0,1.0,15.0
Ireland,23.1,45.6,31.1,17.4,0.0,2.0,15.0,49.2
USA,44.2,100.0,3.0,3.1,69.5,0.9,26.1,2.1


### 6.3 The splits are valid
No station may appear in more than one of train / validation / test, and the file must be small enough for GitHub (limit 100 MB, we stay under 50 MB).

In [11]:
sites = {k: set(final.loc[final["split"] == k, "site_id"]) for k in ["train", "val", "test"]}
assert not (sites["train"] & sites["val"]) and not (sites["train"] & sites["test"]) and not (sites["val"] & sites["test"]), "station leakage!"
assert os.path.getsize(OUT_PATH) < 50e6

print("stations shared between sets:", 0)
for col in ["split", "split_time"]:
    counts = final[col].value_counts()
    print(f"{col:11s}:", {k: f"{counts[k]:,} ({100 * counts[k] / len(final):.0f}%)" for k in ["train", "val", "test"]})
print(f"file size  : {os.path.getsize(OUT_PATH) / 1e6:.1f} MB")

stations shared between sets: 0
split      : {'train': '181,747 (70%)', 'val': '26,054 (10%)', 'test': '51,973 (20%)'}
split_time : {'train': '185,094 (71%)', 'val': '21,499 (8%)', 'test': '53,181 (20%)'}
file size  : 34.7 MB


## 7. Summary and hand-off

- `data/processed/final_dataset.csv` is the **only** processed dataset. The split is stored inside it: `split` (by station, main: train / val / test = 70 / 10 / 20) and `split_time` (by year, secondary: train up to 2015, val 2016-2017, test 2018+).
- All 8 parameters are kept in the file for EDA, but **the models only use dissolved oxygen, pH and temperature** (plus context and history), because the target is calculated from the 8 parameters. See `03_features.ipynb` for the test that proves this.
- Next: `01_eda.ipynb` (explore) and `03_features.ipynb` (feature engineering).